# Comparison: fine-tuned vs. EmbeddingGemma 2 embeddings

Compares the key results of `04_evaluation_gemma.ipynb` across the embeddings used for `emb_*`, `pca_*` and `similarity_cluster_*`:

| run | embeddings |
| --- | --- |
| `finetuned` | fine-tuned DoubleMLDeep text + image embeddings (256 dim.), as in `04_evaluation.ipynb` (re-run in the same environment) |
| `embeddinggemma2_txtimg_dim256` | EmbeddingGemma 2, text + image, Matryoshka 256 dim. (**main specification**) |
| `embeddinggemma2_txt_dim256` | EmbeddingGemma 2, text only, Matryoshka 256 dim. |
| `embeddinggemma2_txtimg_dim768` | EmbeddingGemma 2, text + image, full 768 dim. |

Note that the cluster-similarity variables are not comparable one-to-one across embeddings: the five clusters are re-estimated (k-means) for every embedding, so `similarity_cluster_k` refers to a different group of products in each run.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

palette = sns.color_palette("colorblind")
results_dir = "../output/04_evaluation_gemma"
output_dir = "../output/04_evaluation_gemma/comparison"
Path(output_dir).mkdir(parents=True, exist_ok=True)

runs = {
    "finetuned": "Fine-tuned (txt+img, 256d)",
    "embeddinggemma2_txtimg_dim256": "EmbeddingGemma 2 (txt+img, 256d)",
    "embeddinggemma2_txt_dim256": "EmbeddingGemma 2 (txt, 256d)",
    "embeddinggemma2_txtimg_dim768": "EmbeddingGemma 2 (txt+img, 768d)",
}
runs = {k: v for k, v in runs.items() if os.path.exists(os.path.join(results_dir, k, "key_results.json"))}
print(runs)


def load(file_name, **kwargs):
    frames = []
    for run, label in runs.items():
        df = pd.read_csv(os.path.join(results_dir, run, file_name), **kwargs)
        df.insert(0, "embedding", label)
        frames.append(df)
    return pd.concat(frames)


key_results = {
    run: json.load(open(os.path.join(results_dir, run, "key_results.json")))
    for run in runs
}

## Reproduction check

The `finetuned` run re-executes the original specification in the environment used for the EmbeddingGemma 2 runs. Its estimates are compared with the outputs stored in `04_evaluation.ipynb`.

In [ ]:
# values printed in the committed outputs of 04_evaluation.ipynb: (value, tolerance given the printed precision)
reference_04 = {
    "Linear lagged": (-0.690266, 5e-7),
    "Linear lagged_and_embeddings": (-0.656960, 5e-7),
    "Linear lagged_and_add_controls": (-0.746364, 5e-7),
    "Linear lagged_and_add_controls_and_emb": (-0.712354, 5e-7),
    "Linear lagged_and_add_controls_and_sim": (-0.722537, 5e-7),
    "plr_embeddings": (-0.623, 5e-4),
    "plr_add_controls": (-0.690, 5e-4),
    "plr_embeddings_add_controls": (-0.697126, 5e-7),
    "plr_sim_add_controls": (-0.691, 5e-4),
    "plr_pca_add_controls": (-0.677, 5e-4),
    "R2 outcome plr_embeddings": (0.8636173876939671, 1e-12),
    "R2 treatment plr_embeddings": (0.9730706950762325, 1e-12),
    "R2 outcome plr_embeddings_add_controls": (0.8910037311081709, 1e-12),
    "R2 treatment plr_embeddings_add_controls": (0.9744150172234399, 1e-12),
    "chi2 p-value clusters: Linear Specification": (0.029984178246390436, 1e-12),
    "chi2 p-value clusters: Linear Specification (Interactions)": (0.053882855916301486, 1e-12),
    "chi2 p-value clusters: PLR Specification": (0.0009061660603696398, 1e-12),
    "sensitivity theta lower (cf=0.03, rho=1)": (-0.829018, 5e-7),
    "sensitivity theta upper (cf=0.03, rho=1)": (-0.565233, 5e-7),
    "sensitivity RV": (0.14856088, 5e-9),
}
if "finetuned" in runs:
    ft_dir = os.path.join(results_dir, "finetuned")
    ft_plr = pd.read_csv(os.path.join(ft_dir, "plr_ate_estimates.csv")).set_index("model")
    ft_chi2 = pd.read_csv(os.path.join(ft_dir, "chi2_tests.csv")).set_index("model")
    ft_sens = key_results["finetuned"]["sensitivity"]
    rerun = pd.concat(
        [
            pd.read_csv(os.path.join(ft_dir, "linear_ate_estimates.csv")).set_index("model")["coef"],
            ft_plr["coef"],
        ]
    ).to_dict()
    for m in ["plr_embeddings", "plr_embeddings_add_controls"]:
        rerun[f"R2 outcome {m}"] = ft_plr.loc[m, "r2_outcome"]
        rerun[f"R2 treatment {m}"] = ft_plr.loc[m, "r2_treatment"]
    for m in ft_chi2.index:
        rerun[f"chi2 p-value clusters: {m}"] = ft_chi2.loc[m, "p_value_clusters"]
    rerun["sensitivity theta lower (cf=0.03, rho=1)"] = ft_sens["theta_lower_cf003"]
    rerun["sensitivity theta upper (cf=0.03, rho=1)"] = ft_sens["theta_upper_cf003"]
    rerun["sensitivity RV"] = ft_sens["rv"]

    df_repro = pd.DataFrame(reference_04, index=["04_evaluation.ipynb", "tolerance"]).T
    df_repro["re-run"] = pd.Series(rerun)
    df_repro["abs. difference"] = (df_repro["re-run"] - df_repro["04_evaluation.ipynb"]).abs()
    df_repro["reproduced"] = df_repro["abs. difference"] <= df_repro["tolerance"]
    print(f"Reproduced {df_repro['reproduced'].sum()} of {len(df_repro)} reference values")
    display(df_repro)

## Average effects

In [ ]:
df_linear = load("linear_ate_estimates.csv")
df_plr = load("plr_ate_estimates.csv")
df_ate = pd.concat([df_linear, df_plr])
df_ate["estimate (90% CI)"] = df_ate.apply(
    lambda r: f"{r['coef']:.3f} [{r['lower']:.3f}, {r['upper']:.3f}]", axis=1
)
df_ate_table = df_ate.pivot(index="model", columns="embedding", values="estimate (90% CI)")
df_ate_table = df_ate_table.loc[df_ate["model"].unique(), list(runs.values())]
df_ate_table.to_csv(os.path.join(output_dir, "ate_estimates.csv"))
df_ate_table

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, df, title in [
    (axs[0], df_linear[df_linear["model"] != "Linear interactions"], "Linear models"),
    (axs[1], df_plr, "PLR models (boosted trees)"),
]:
    models = list(df["model"].unique())
    x = np.arange(len(models))
    width = 0.8 / len(runs)
    for i, label in enumerate(runs.values()):
        d = df[df["embedding"] == label].set_index("model").loc[models]
        ax.errorbar(
            x + (i - (len(runs) - 1) / 2) * width,
            d["coef"],
            yerr=[d["coef"] - d["lower"], d["upper"] - d["coef"]],
            fmt="o",
            capsize=4,
            color=palette[i],
            label=label,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha="right")
    ax.set_title(title)
    ax.grid(True)
axs[0].set_ylabel("Estimated ATE (90% CI)")
axs[1].legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "ate_estimates_comparison.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Nuisance fit of the PLR models (cross-fitted R^2)
df_r2 = df_plr.pivot(index="model", columns="embedding", values="r2_outcome").loc[df_plr["model"].unique(), list(runs.values())]
df_r2_d = df_plr.pivot(index="model", columns="embedding", values="r2_treatment").loc[df_plr["model"].unique(), list(runs.values())]
print("R^2 outcome learner")
display(df_r2.round(4))
print("R^2 treatment learner")
display(df_r2_d.round(4))

## GATEs by most similar cluster

In [ ]:
df_gates = load("gate_estimates.csv", index_col=0).reset_index(names="cluster")
df_gates["estimate (95% CI)"] = df_gates.apply(
    lambda r: f"{r['coef']:.3f} [{r['[0.025']:.3f}, {r['0.975]']:.3f}] (n={r['n_obs']})", axis=1
)
df_gates.pivot(index="cluster", columns="embedding", values="estimate (95% CI)")[list(runs.values())]

## CATEs (best linear predictor)

In [ ]:
df_cate = load("cate_estimates.csv")
df_cate["estimate (90% CI)"] = df_cate.apply(
    lambda r: f"{r['coef']:.3f} [{r['lower']:.3f}, {r['upper']:.3f}]", axis=1
)
for model in df_cate["model"].unique():
    print(model)
    d = df_cate[df_cate["model"] == model]
    display(d.pivot(index="variable", columns="embedding", values="estimate (90% CI)").loc[d["variable"].unique(), list(runs.values())])

In [ ]:
df_chi2 = load("chi2_tests.csv")
print("Chi^2 tests: p-values for H0 'no heterogeneity along the cluster similarities' / 'no heterogeneity at all'")
display(df_chi2.pivot(index="model", columns="embedding", values="p_value_clusters")[list(runs.values())])
display(df_chi2.pivot(index="model", columns="embedding", values="p_value_heterogeneity")[list(runs.values())])

In [ ]:
df_var = load("cate_variance_explained.csv", index_col=0).reset_index(names="regressors")
df_var.melt(id_vars=["embedding", "regressors"], var_name="model", value_name="R2").pivot(
    index=["model", "regressors"], columns="embedding", values="R2"
)[list(runs.values())].round(3)

In [ ]:
# Distribution of the per-observation CATEs
fig, axs = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
titles = {
    "SortEff_lin": "Linear Specification",
    "SortEff_tab_dml": "Linear Specification (Interactions)",
    "SortEff_deep_dml": "PLR Specification (Boosted Trees)",
}
sorted_effects = {
    label: pd.read_csv(os.path.join(results_dir, run, "sorted_effects.csv"))
    for run, label in runs.items()
}
for ax, (col, title) in zip(axs, titles.items()):
    for i, (label, df) in enumerate(sorted_effects.items()):
        values = np.sort(df[col].values)
        ax.plot(np.linspace(0, 1, len(values)), values, color=palette[i], label=label)
    ax.set_title(title)
    ax.set_xlabel("Percentile index")
    ax.grid(True)
axs[0].set_ylabel("Sorted CATE")
axs[0].legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "sorted_effects_comparison.png"), bbox_inches="tight")
plt.show()

pd.DataFrame(
    {
        (label, col): df[col].quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        for label, df in sorted_effects.items()
        for col in titles
    }
).T.round(3)

In [ ]:
# Correlation of the per-observation CATEs (PLR specification) across embeddings
df_corr = pd.DataFrame({label: df["SortEff_deep_dml"].values for label, df in sorted_effects.items()})
df_corr.corr().round(3)

## Sensitivity analysis (PLR with embeddings and additional controls)

In [ ]:
df_sens = pd.DataFrame({runs[run]: res["sensitivity"] for run, res in key_results.items()})
df_sens.to_csv(os.path.join(output_dir, "sensitivity_comparison.csv"))
df_sens.round(4)

In [ ]:
df_sensitivity = load("sensitivity_analysis.csv")
df_sensitivity["bounds [CI]"] = df_sensitivity.apply(
    lambda r: f"[{r['ci_lower']:.3f}] {r['theta_lower']:.3f} .. {r['theta_upper']:.3f} [{r['ci_upper']:.3f}]", axis=1
)
df_sensitivity.pivot(index=["rho", "R2_y"], columns="embedding", values="bounds [CI]")[list(runs.values())]